# DLC Pupil Tracking QC Validation

Quantitative validation of DeepLabCut landmark accuracy **(Part A)** and pupil ellipse-fit quality **(Parts B–G)**.

**Lizard notes:** analyzed H5 outputs use iteration-0 / snapshot **900k**; landmark evaluation uses iteration-1 / snapshot **950k**. PV24 source videos are absent from the migrated copy (labels only).

Exports land in `outputs/dlc_qc_<species>/`.

## 0. Configuration

In [ ]:
%matplotlib inline
from pathlib import Path

REPO = Path.cwd().resolve()
if not (REPO / "src" / "eye_tracking_system_tools").is_dir():
    REPO = REPO.parent  # running from development/

# --- user parameters ---
DLC_PROJECT_ROOT = Path("/Volumes/Data/Nimrod/DLC_migration_here/Eye_Tracking_pipline-Nimrod-2021-03-01")
SPECIES = "lizard"
ITERATION = 1          # for landmark eval / labeled data
SHUFFLE = 1
LIKELIHOOD_P_CUTOFF = 0.6   # DLC config pcutoff; tune here
MIN_POINTS_FOR_ELLIPSE = 6  # >=5 mathematically; 6 matches BlockSync
REPRESENTATIVE_DIAMETER = "geometric_mean"  # 2*sqrt(major*minor)
VIDEOS_DIR = None  # default: DLC_PROJECT_ROOT / "videos"

OUTPUT_DIR = REPO / "outputs" / f"dlc_qc_{SPECIES}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("REPO:", REPO)
print("DLC:", DLC_PROJECT_ROOT)
print("OUT:", OUTPUT_DIR)

## 1. Run full QC pipeline (Parts A–G)

In [ ]:
from eye_tracking_system_tools.analysis.dlc_validation import run_full_qc_pipeline

result = run_full_qc_pipeline(
    DLC_PROJECT_ROOT,
    OUTPUT_DIR,
    species=SPECIES,
    iteration=ITERATION,
    shuffle=SHUFFLE,
    likelihood_p_cutoff=LIKELIHOOD_P_CUTOFF,
    min_points_for_ellipse=MIN_POINTS_FOR_ELLIPSE,
    diameter_method=REPRESENTATIVE_DIAMETER,
    videos_dir=VIDEOS_DIR,
    show_progress=True,
)

project = result["project"]
print("Pupil bodyparts:", project.pupil_bodyparts)
print("\n" + result["summary_text"])

## 2. Inspect landmark validation (Part A)

In [ ]:
lr = result.get("landmark_result")
if lr:
    display(lr["native_summary"])
    display(lr["summary"].query("filter != 'per_bodypart'"))
    display(lr["frame_errors"].head())
else:
    print("Landmark validation not available.")

## 3. Inspect ellipse QC (Parts B–D)

In [ ]:
er = result["ellipse_result"]
print("Filter stats:", er["filter_stats"])
display(er["frame_df"].head())
for level, df in (result.get("summaries") or {}).items():
    print(f"\n--- {level} ---")
    display(df)

## 4. Figures and visual QC

Diagnostic PDFs: `OUTPUT_DIR/figures/`. Representative frames: `OUTPUT_DIR/figures/qc_frames/`.

In [ ]:
from IPython.display import Image, display as ipy_display

fig_dir = OUTPUT_DIR / "figures"
for pdf in sorted(fig_dir.glob("*.pdf")):
    print(pdf.name)

qc_root = fig_dir / "qc_frames"
if qc_root.is_dir():
    for png in sorted(qc_root.rglob("*.png"))[:6]:
        print(png.relative_to(OUTPUT_DIR))
        ipy_display(Image(filename=str(png), width=480))

## 5. Export paths

In [ ]:
for name, path in result["export_paths"].items():
    print(f"{name}: {path}")
print("config:", result["config_path"])